# 모기 비행 궤적 예측 — sub_24 완전 재현 v2

## 목표
**학습부터 제출까지 완전히 동일한 sub_24 (LB 0.6780) 재현**.

## v2 변경 사항
1. **결정성 모드** — cuDNN deterministic, seed 고정
2. **Calibration α 하드코드** — (1, 0.95, 1), grid search 제거
3. **Drive 백업 cross-check + fallback** — 100% 재현 보장 옵션

## v1 재현 실패 원인 (LB 0.6756)
- 셋업 A OOF 0.6591 (vs 원본 0.6608): 학습 비결정성 fluctuation
- sub_09 OOF 0.6598 (vs 0.6612)
- ★ Calibration grid search가 다른 α 발견 (0.9, 0.825, 1) — overfit

## v2 사용 모드
| 모드 | 보장 | 시간 |
|---|---|---|
| **A (권고): 학습 + Drive 백업 사용** | LB 0.6780 100% | ~1.5시간 |
| B: 학습 + α 하드코드 결과 사용 | LB 0.6776~0.6780 | ~1.5시간 |
| C: 학습 skip + 백업만 사용 | LB 0.6780 100% | ~2분 |

셀 12에서 `USE_DRIVE_BACKUP` 플래그로 선택.


In [1]:
import os, subprocess, matplotlib as mpl
if not os.path.exists("/usr/share/fonts/truetype/nanum"):
    subprocess.run(["apt-get", "-qq", "install", "fonts-nanum"], check=False)
    subprocess.run(["fc-cache", "-fv"], check=False)
mpl.rcParams["font.family"] = "NanumGothic"
mpl.rcParams["axes.unicode_minus"] = False

import numpy as np
import pandas as pd

# ============================================================
# ★ v2: 결정성 모드 (numpy/random/python hash)
# ============================================================
import random
os.environ["PYTHONHASHSEED"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(0); np.random.seed(0)

print("환경 + 결정성 모드 (numpy/random) 적용")


환경 + 결정성 모드 (numpy/random) 적용


## 1. 데이터 로드 (Drive + zip + CSV)


In [2]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH    = '/content/drive/MyDrive/competition/mosquito/open.zip'
EXTRACT_DIR = '/content/data'
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.listdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('압축 해제 완료')
else:
    print('이미 압축 해제됨 → 스킵')

Mounted at /content/drive
압축 해제 완료


In [3]:
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR        = '/content/data'
TRAIN_DIR       = os.path.join(DATA_DIR, 'train')
TEST_DIR        = os.path.join(DATA_DIR, 'test')
LABELS_PATH     = os.path.join(DATA_DIR, 'train_labels.csv')
SUBMISSION_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.csv')))
test_files  = sorted(glob.glob(os.path.join(TEST_DIR,  '*.csv')))
labels      = pd.read_csv(LABELS_PATH)
sub         = pd.read_csv(SUBMISSION_PATH)

def load_stack(files, desc):
    arrays = []
    for f in tqdm(files, desc=desc):
        df = pd.read_csv(f)
        arrays.append(df[['x','y','z']].values)
    return np.stack(arrays, axis=0).astype(np.float64)

X_train = load_stack(train_files, 'train')
X_test  = load_stack(test_files,  'test')

# ID 매칭 + y_train 정렬
train_ids = [os.path.splitext(os.path.basename(f))[0] for f in train_files]
test_ids  = [os.path.splitext(os.path.basename(f))[0] for f in test_files]
y_train   = labels.set_index('id').loc[train_ids][['x','y','z']].values.astype(np.float64)

print(f'X_train {X_train.shape}, y_train {y_train.shape}')
print(f'X_test  {X_test.shape}, sub {sub.shape}')
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0]  == len(sub)

train:   0%|          | 0/10000 [00:00<?, ?it/s]

test:   0%|          | 0/10000 [00:00<?, ?it/s]

X_train (10000, 11, 3), y_train (10000, 3)
X_test  (10000, 11, 3), sub (10000, 4)


## 2. 칼만 필터 (Constant Velocity) 예측

σ_obs=0.3mm, σ_proc=1.0 (그리드 서치 best). LB 0.6452 baseline.


In [4]:
import torch

DT     = 0.040                          # 40 ms
T_PRED = 0.080                          # 80 ms 외삽
t_obs  = np.arange(-400, 1, 40) / 1000.0  # (-0.4, ..., 0.0) sec
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def cos_safe(a, b):
    """안전한 코사인 유사도 (벡터 norm 0 보호)."""
    na = np.linalg.norm(a, axis=-1); nb = np.linalg.norm(b, axis=-1)
    denom = np.maximum(na * nb, 1e-12)
    return np.clip((a*b).sum(-1) / denom, -1, 1)

def r_hit_metric(pred_3d, true_3d, thr=0.01):
    """R-Hit@1cm + 거리 배열 반환."""
    d = np.linalg.norm(pred_3d - true_3d, axis=-1)
    return float((d <= thr).mean()), d

print(f'device: {device}, DT={DT}, T_PRED={T_PRED}')

# ============================================================
# ★ v2: torch 결정성 모드
# ============================================================
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    print("★ torch deterministic: cudnn.deterministic=True, benchmark=False")
except Exception as e:
    print(f"deterministic warning: {e}")


device: cuda, DT=0.04, T_PRED=0.08
★ torch deterministic: cudnn.deterministic=True, benchmark=False


In [5]:
def kalman_predict(X, model='CV', dt=DT, t_pred=T_PRED,
                   sigma_obs=0.001, sigma_proc=5.0, P0=1.0):
    """각 축 독립 벡터화 칼만 필터 + t_pred 외삽."""
    N, T, _ = X.shape
    if model == 'CV':
        F      = np.array([[1, dt], [0, 1]])
        F_pred = np.array([[1, t_pred], [0, 1]])
        Q      = sigma_proc**2 * np.array([[dt**4/4, dt**3/2],
                                           [dt**3/2, dt**2]])
        n_state = 2
    else:
        F      = np.array([[1, dt, dt**2/2], [0, 1, dt], [0, 0, 1]])
        F_pred = np.array([[1, t_pred, t_pred**2/2], [0, 1, t_pred], [0, 0, 1]])
        Q      = sigma_proc**2 * np.array([[dt**4/4, dt**3/2, dt**2/2],
                                           [dt**3/2, dt**2,   dt    ],
                                           [dt**2/2, dt,      1     ]])
        n_state = 3
    R = sigma_obs**2
    pred = np.zeros((N, 3))
    for j in range(3):
        z_all = X[:, :, j]
        state = np.zeros((N, n_state))
        state[:, 0] = z_all[:, 0]
        P = np.eye(n_state) * P0
        for t in range(1, T):
            state = state @ F.T
            P     = F @ P @ F.T + Q
            innov = z_all[:, t] - state[:, 0]
            S     = P[0, 0] + R
            K     = P[:, 0] / S
            state = state + np.outer(innov, K)
            P     = P - np.outer(K, P[0, :])
        pred[:, j] = (state @ F_pred.T)[:, 0]
    return pred

In [6]:
SO_BEST, SP_BEST = 0.30e-3, 1.0
print('칼만 예측 생성 중...')
kalman_train = kalman_predict(X_train, 'CV', sigma_obs=SO_BEST, sigma_proc=SP_BEST)
kalman_test  = kalman_predict(X_test,  'CV', sigma_obs=SO_BEST, sigma_proc=SP_BEST)
res_target = y_train - kalman_train

rh_kal_train, _ = r_hit_metric(kalman_train, y_train)
print(f'칼만 train R-Hit = {rh_kal_train:.4f}  (이전 측정 0.5964)')
assert abs(rh_kal_train - 0.5964) < 0.0005, f'칼만 재현 실패: {rh_kal_train}'

칼만 예측 생성 중...
칼만 train R-Hit = 0.5964  (이전 측정 0.5964)


## 3. 노이즈 추정 + 스칼라 features (21차원)

CubicSpline + LOO로 노이즈 추정. log1p 변환된 21개 통계 features.


In [7]:
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter

def noise_poly2(X, T_obs=t_obs):
    V    = np.vander(T_obs, 3, increasing=False)
    out  = np.zeros(X.shape[0])
    for j in range(3):
        coef = np.linalg.lstsq(V, X[:, :, j].T, rcond=None)[0]
        fit  = (V @ coef).T
        out += (X[:, :, j] - fit).std(axis=1)
    return out / 3

def noise_savgol(X, w=5, p=2):
    Xs = savgol_filter(X, window_length=w, polyorder=p, axis=1)
    return (X - Xs).std(axis=1).mean(axis=-1)

def noise_loo_spline(X, T_obs=t_obs):
    N, T, _ = X.shape; out = np.zeros(N)
    idx_all = np.arange(T)
    for i in tqdm(range(N), desc='LOO spline'):
        s = 0.0
        for k in range(1, T-1):
            mask = idx_all != k
            for j in range(3):
                cs = CubicSpline(T_obs[mask], X[i, mask, j])
                s += (X[i, k, j] - cs(T_obs[k]))**2
        out[i] = np.sqrt(s / ((T-2) * 3))
    return out

DOCS_DIR = '/content/drive/MyDrive/competition/mosquito/docs'
os.makedirs(DOCS_DIR, exist_ok=True)
NOISE_CACHE = f'{DOCS_DIR}/noise_cache.npz'

if os.path.exists(NOISE_CACHE):
    nc = np.load(NOISE_CACHE)
    noise_p, noise_s, noise_l = nc['noise_p'], nc['noise_s'], nc['noise_l']
    noise_p_test, noise_s_test = nc['noise_p_test'], nc['noise_s_test']
    print(f'노이즈 캐시 로드')
else:
    print('노이즈 추정 새로 계산...')
    noise_p = noise_poly2(X_train)
    noise_s = noise_savgol(X_train)
    noise_l = noise_loo_spline(X_train)
    noise_p_test = noise_poly2(X_test)
    noise_s_test = noise_savgol(X_test)
    np.savez(NOISE_CACHE,
             noise_p=noise_p, noise_s=noise_s, noise_l=noise_l,
             noise_p_test=noise_p_test, noise_s_test=noise_s_test)
    print(f'캐시 저장: {NOISE_CACHE}')

노이즈 캐시 로드


In [8]:
def build_scalar_feats(X, noise_p_arr, noise_s_arr, noise_loo_arr=None):
    """X: (N, 11, 3) → 21 변수 DataFrame. test에서는 noise_loo를 noise_savgol로 대체."""
    delta_ = np.diff(X, axis=1)
    v_     = delta_ / DT
    a_     = np.diff(v_, axis=1) / DT
    jerk_  = np.diff(a_, axis=1) / DT
    sp_    = np.linalg.norm(v_, axis=-1)
    ac_    = np.linalg.norm(a_, axis=-1)
    jk_    = np.linalg.norm(jerk_, axis=-1)
    v_l    = v_[:, -1, :]
    a_l    = a_[:, -1, :]
    a_r    = a_[:, -3:, :].mean(axis=1)
    nd_vec = X[:, -1] - X[:, 0]
    nd     = np.linalg.norm(nd_vec, axis=-1)
    pl     = np.linalg.norm(delta_, axis=-1).sum(axis=1)
    straight = np.where(pl > 1e-12, nd / np.maximum(pl, 1e-12), 0.0)
    turn   = cos_safe(v_l, v_[:, :-1, :].mean(axis=1))
    if noise_loo_arr is None:
        noise_loo_arr = noise_s_arr
    return pd.DataFrame({
        'mean_speed' : sp_.mean(1),    'max_speed'  : sp_.max(1),
        'speed_std'  : sp_.std(1),     'mean_acc'   : ac_.mean(1),
        'max_acc'    : ac_.max(1),     'max_jerk'   : jk_.max(1),
        'straightness': straight,      'net_disp'   : nd,
        'turn_cos'   : turn,           '|v_last|'   : np.linalg.norm(v_l, axis=-1),
        '|a_last|'   : np.linalg.norm(a_l, axis=-1),
        '|a_recent|' : np.linalg.norm(a_r, axis=-1),
        'jerk_last'  : jk_[:, -1],     'jerk_recent': jk_[:, -3:].mean(1),
        'noise_poly2'  : noise_p_arr,
        'noise_savgol' : noise_s_arr,
        'noise_loo'    : noise_loo_arr,
        'hard_turn'  : (turn < 0.5).astype(np.float32),
        'high_speed' : (np.linalg.norm(v_l, axis=-1) > 1.0).astype(np.float32),
        'high_acc'   : (ac_.max(axis=1) > 15).astype(np.float32),
        'log_max_acc': np.log1p(ac_.max(1)),
    })

scal_train = build_scalar_feats(X_train, noise_p, noise_s, noise_l)
scal_test  = build_scalar_feats(X_test,  noise_p_test, noise_s_test)
SCAL_COLS_BASE = scal_train.columns.tolist()
assert len(SCAL_COLS_BASE) == 21

# log1p 적용 (long-tail)
LOG_COLS = ['mean_speed','max_speed','speed_std','mean_acc','max_acc','max_jerk',
            'net_disp','|v_last|','|a_last|','|a_recent|','jerk_last','jerk_recent',
            'noise_poly2','noise_savgol','noise_loo']
for c in LOG_COLS:
    scal_train[c] = np.log1p(scal_train[c])
    scal_test[c]  = np.log1p(scal_test[c])

X_scal_base_train = scal_train[SCAL_COLS_BASE].values.astype(np.float32)
X_scal_base_test  = scal_test[SCAL_COLS_BASE].values.astype(np.float32)
print(f'X_scal_base_train: {X_scal_base_train.shape} (21 변수, log1p 적용)')

X_scal_base_train: (10000, 21) (21 변수, log1p 적용)


## 4. Yaw 회전 + T+8 타깃 정의

마지막 velocity 방향으로 yaw 정규화 (회전 좌표계). T+8 = +80ms 위치 잔차 (회전 좌표계).


In [9]:
def yaw_angle(v):
    return np.arctan2(v[:, 1], v[:, 0])

def rotate_xy(vec, theta):
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([
         vec[:, 0] * c + vec[:, 1] * s,
        -vec[:, 0] * s + vec[:, 1] * c,
         vec[:, 2],
    ], axis=-1)

def inverse_rotate_xy(vec, theta):
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([
        vec[:, 0] * c - vec[:, 1] * s,
        vec[:, 0] * s + vec[:, 1] * c,
        vec[:, 2],
    ], axis=-1)

# 검증
np.random.seed(0)
test_vec   = np.random.randn(100, 3)
test_theta = np.random.uniform(-np.pi, np.pi, 100)
err = np.abs(inverse_rotate_xy(rotate_xy(test_vec, test_theta), test_theta) - test_vec).max()
assert err < 1e-12, f'회전 항등성 실패: {err}'
print(f'회전 유틸 검증 통과 (max err = {err:.2e})')

회전 유틸 검증 통과 (max err = 4.44e-16)


In [10]:
v_last_train = (X_train[:, -1] - X_train[:, -2]) / DT
v_last_test  = (X_test[:, -1]  - X_test[:, -2])  / DT
theta_train  = yaw_angle(v_last_train)
theta_test   = yaw_angle(v_last_test)

target_T1 = y_train - kalman_train             # 칼만 잔차 (m)
target_T8 = rotate_xy(target_T1, theta_train)  # 회전된 잔차 (m) — 베스트 타깃

# EDA 일치 검증 (회전 정의 정확성)
expected = -0.36
actual = target_T8[:, 0].mean() * 100
assert abs(actual - expected) < 0.05, f'T-8 검증 실패: {actual} (예상 {expected})'
print(f'T-8 검증 통과: x mean = {actual:+.4f} cm (EDA -0.36)')

T-8 검증 통과: x mean = -0.3603 cm (EDA -0.36)


## 5. Tier 3 features (40차원 스칼라)

21차원 기본 스칼라 + 19차원 Tier 3 (rolling speed × DT + cumulative path) = 40차원.


In [11]:
def build_disp_t3(X):
    """Tier 3 disp 입력 — (N, 27, 4).

    27 시점 = disp(10) + a×DT²(9) + j×DT³(8)
    4 채널  = (x, y, z, norm)
    모두 m 단위로 통일.
    """
    disp = np.diff(X, axis=1)              # (N, 10, 3)
    v    = disp / DT
    a    = np.diff(v, axis=1) / DT
    j    = np.diff(a, axis=1) / DT
    a_scaled = a * (DT ** 2)               # (N, 9, 3) — m
    j_scaled = j * (DT ** 3)               # (N, 8, 3) — m

    disp_xyz = np.concatenate([disp, a_scaled, j_scaled], axis=1)   # (N, 27, 3)
    norm_ext = np.linalg.norm(disp_xyz, axis=-1, keepdims=True)     # (N, 27, 1)
    disp_t3  = np.concatenate([disp_xyz, norm_ext], axis=-1)        # (N, 27, 4)
    return disp_t3.astype(np.float32)

def build_tier3_extra(X):
    """Tier 3 추가 스칼라 — (N, 19).

    19 변수 = rolling_speed_mean × DT (8) + cumulative_path_length (11)
    모두 m 단위.
    """
    disp  = np.diff(X, axis=1)
    v     = disp / DT
    speed = np.linalg.norm(v, axis=-1)     # (N, 10) — m/s

    # rolling speed mean (window=3) → 8 시점
    speed_roll = np.stack([
        speed[:, i:i+3].mean(axis=1) for i in range(8)
    ], axis=1) * DT                         # (N, 8) — m

    # cumulative path length
    disp_norm = np.linalg.norm(disp, axis=-1)
    cum_path = np.concatenate([
        np.zeros((X.shape[0], 1)),
        np.cumsum(disp_norm, axis=1),
    ], axis=1)                               # (N, 11) — m

    return np.concatenate([speed_roll, cum_path], axis=1).astype(np.float32)

# 입력 텐서 생성
rel_train_t3  = (X_train - X_train[:, -1:, :]).astype(np.float32)
rel_test_t3   = (X_test  - X_test[:,  -1:, :]).astype(np.float32)
disp_train_t3 = build_disp_t3(X_train)
disp_test_t3  = build_disp_t3(X_test)
tier3_train   = build_tier3_extra(X_train)
tier3_test    = build_tier3_extra(X_test)

# X_scal 합치기 (40차원)
X_scal_train_t3 = np.concatenate([X_scal_base_train, tier3_train], axis=-1).astype(np.float32)
X_scal_test_t3  = np.concatenate([X_scal_base_test,  tier3_test], axis=-1).astype(np.float32)

print(f'rel_train_t3   : {rel_train_t3.shape}')
print(f'disp_train_t3  : {disp_train_t3.shape}  (27 시점, 4 채널)')
print(f'X_scal_train_t3: {X_scal_train_t3.shape}  (21 기본 + 19 Tier 3 = 40)')
print(f'  → flatten 차원: 33 + {27*4} + 40 = {33 + 27*4 + 40}')

# 안정성 검증
print(f'\n=== Tier 3 입력 안정성 ===')
for name, arr in [('disp t3 [0:10]',  disp_train_t3[:, :10, :3]),
                   ('disp t3 [10:19] (a×DT²)', disp_train_t3[:, 10:19, :3]),
                   ('disp t3 [19:27] (j×DT³)', disp_train_t3[:, 19:, :3]),
                   ('disp t3 norm 채널', disp_train_t3[:, :, 3]),
                   ('X_scal t3 추가 [21:29] (speed_roll)', tier3_train[:, :8]),
                   ('X_scal t3 추가 [29:40] (cum_path)',   tier3_train[:, 8:])]:
    print(f'  {name:<35}: std={arr.std():.5f}, '
          f'range=[{arr.min():+.4f}, {arr.max():+.4f}]')

rel_train_t3   : (10000, 11, 3)
disp_train_t3  : (10000, 27, 4)  (27 시점, 4 채널)
X_scal_train_t3: (10000, 40)  (21 기본 + 19 Tier 3 = 40)
  → flatten 차원: 33 + 108 + 40 = 181

=== Tier 3 입력 안정성 ===
  disp t3 [0:10]                     : std=0.01584, range=[-0.0538, +0.0539]
  disp t3 [10:19] (a×DT²)            : std=0.00524, range=[-0.0890, +0.0781]
  disp t3 [19:27] (j×DT³)            : std=0.00675, range=[-0.1082, +0.1271]
  disp t3 norm 채널                    : std=0.01387, range=[+0.0000, +0.1398]
  X_scal t3 추가 [21:29] (speed_roll)  : std=0.01352, range=[+0.0000, +0.0540]
  X_scal t3 추가 [29:40] (cum_path)    : std=0.10939, range=[+0.0000, +0.5395]


## 6. 손실 함수 + F+W 보조 타깃

- 메인 손실: `combo = euclid + 0.3 × softhit` (R-Hit@1cm surrogate)
- F 보조 타깃: T+7 직접 변위 (회전 좌표계)
- W 보조 타깃: 다른 σ 칼만 잔차 (σ_obs=1mm, 3.3배)


In [12]:
import torch.nn as nn
import torch.nn.functional as F

def loss_mse(pred, true):    return F.mse_loss(pred, true)
def loss_mae(pred, true):    return F.l1_loss(pred, true)
def loss_huber(pred, true):  return F.huber_loss(pred, true, delta=0.005)
def loss_euclid(pred, true):
    return torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12).mean()
def loss_softhit(pred, true, beta=0.002):
    d = torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12)
    return torch.sigmoid((d - 0.01) / beta).mean()
def loss_combo(pred, true):
    return loss_euclid(pred, true) + 0.3 * loss_softhit(pred, true)
def loss_combo_05(pred, true):
    return loss_euclid(pred, true) + 0.5 * loss_softhit(pred, true)

LOSSES_V2 = {
    'mae': loss_mae, 'huber': loss_huber, 'euclid': loss_euclid,
    'softhit': loss_softhit, 'combo': loss_combo, 'combo05': loss_combo_05,
    'mse': loss_mse,
}
print(f'손실 함수 {len(LOSSES_V2)}종')

손실 함수 7종


In [13]:
# T-7 (직접 변위, 회전 좌표계)
direct_disp_global = y_train - X_train[:, -1]
target_T7 = rotate_xy(direct_disp_global, theta_train)
aux_target_F = target_T7
print(f'F 보조 타깃 (T-7, 회전 좌표계 직접 변위):')
print(f'  shape: {aux_target_F.shape}')
for j, ax in enumerate(['forward', 'lateral', 'vertical']):
    a = aux_target_F[:, j] * 100
    print(f'  {ax:<8}: mean={a.mean():+7.3f}cm, std={a.std():.3f}cm, p99={np.percentile(a, 99):+.3f}cm')

# W (다른 σ 칼만 잔차, 회전 좌표계)
SO_ALT, SP_ALT = 1.0e-3, 1.0   # 베스트 σo=0.3mm의 3.3배
print(f'\nW 보조 타깃: 다른 σ 칼만 (σo={SO_ALT*1000}mm, σp={SP_ALT}) 잔차')
print('다른 σ 칼만 외삽 생성 중...')
kalman_train_alt = kalman_predict(X_train, 'CV', sigma_obs=SO_ALT, sigma_proc=SP_ALT)
target_T8_alt = rotate_xy(y_train - kalman_train_alt, theta_train)
aux_target_W = target_T8_alt

print(f'  shape: {aux_target_W.shape}')
for j, ax in enumerate(['forward', 'lateral', 'vertical']):
    a = aux_target_W[:, j] * 100
    main_std = target_T8[:, j].std() * 100
    aux_std  = aux_target_W[:, j].std() * 100
    corr     = np.corrcoef(target_T8[:, j], aux_target_W[:, j])[0, 1]
    print(f'  {ax:<8}: 메인 std={main_std:.3f}, 보조 std={aux_std:.3f}, 상관={corr:+.4f}')

F 보조 타깃 (T-7, 회전 좌표계 직접 변위):
  shape: (10000, 3)
  forward : mean= +4.231cm, std=2.933cm, p99=+10.709cm
  lateral : mean= +0.007cm, std=1.254cm, p99=+4.394cm
  vertical: mean= +0.291cm, std=2.398cm, p99=+6.342cm

W 보조 타깃: 다른 σ 칼만 (σo=1.0mm, σp=1.0) 잔차
다른 σ 칼만 외삽 생성 중...
  shape: (10000, 3)
  forward : 메인 std=1.416, 보조 std=1.561, 상관=+0.8839
  lateral : 메인 std=1.220, 보조 std=1.464, 상관=+0.8768
  vertical: 메인 std=0.937, 보조 std=1.054, 상관=+0.8678


In [14]:
def loss_aux_mse(pred, target):
    return F.mse_loss(pred, target)

def loss_aux_euclid(pred, target):
    return torch.sqrt(((pred - target) ** 2).sum(dim=-1) + 1e-12).mean()

print('보조 손실 함수: loss_aux_mse, loss_aux_euclid')

보조 손실 함수: loss_aux_mse, loss_aux_euclid


## 7. 시계열 입력 (11 시점 × 9 채널)

- 채널 0~2: 마지막 시점 기준 상대 위치 (rel)
- 채널 3~5: velocity (zero pad @ t=0)
- 채널 6~8: acceleration (zero pad @ t=0, 1)


In [15]:
def build_seq_t3(X):
    """진짜 시계열 (11 시점, 9 채널) — TCN/LSTM 입력용."""
    N = X.shape[0]
    rel = X - X[:, -1:, :]                      # (N, 11, 3)

    disp = np.diff(X, axis=1)                   # (N, 10, 3)
    v = disp / DT                                # (N, 10, 3) m/s
    v_padded = np.concatenate([                  # (N, 11, 3)
        np.zeros((N, 1, 3)), v
    ], axis=1)

    a = np.diff(v, axis=1) / DT                  # (N, 9, 3) m/s²
    a_padded = np.concatenate([                  # (N, 11, 3)
        np.zeros((N, 2, 3)), a
    ], axis=1)

    seq = np.concatenate([rel, v_padded, a_padded], axis=-1)  # (N, 11, 9)
    return seq.astype(np.float32)

seq_train = build_seq_t3(X_train)
seq_test  = build_seq_t3(X_test)

print(f'seq_train: {seq_train.shape}  (11 시점 × 9 채널)')
print(f'seq_test:  {seq_test.shape}')

# 채널별 정규화 (학습 시 fold 안에서 적용 권고, 여기는 검증용)
print(f'\n=== 채널별 안정성 ===')
for ch in range(9):
    arr = seq_train[:, :, ch]
    name = ['rel_x','rel_y','rel_z','v_x','v_y','v_z','a_x','a_y','a_z'][ch]
    print(f'  ch{ch} ({name:<6}): std={arr.std():.4f}, '
          f'range=[{arr.min():+.3f}, {arr.max():+.3f}]')

def normalize_seq(arr, scaler):
    """채널별 정규화. arr: (N, T, C)"""
    N, T, C = arr.shape
    flat = arr.reshape(-1, C)
    flat_scaled = scaler.transform(flat).astype(np.float32)
    return flat_scaled.reshape(N, T, C)

print(f'\nbuild_seq_t3, normalize_seq 정의 완료')

seq_train: (10000, 11, 9)  (11 시점 × 9 채널)
seq_test:  (10000, 11, 9)

=== 채널별 안정성 ===
  ch0 (rel_x ): std=0.0993, range=[-0.534, +0.506]
  ch1 (rel_y ): std=0.0882, range=[-0.521, +0.534]
  ch2 (rel_z ): std=0.0653, range=[-0.464, +0.487]
  ch3 (v_x   ): std=0.4207, range=[-1.306, +1.348]
  ch4 (v_y   ): std=0.3835, range=[-1.345, +1.348]
  ch5 (v_z   ): std=0.2823, range=[-1.340, +1.286]
  ch6 (a_x   ): std=3.3692, range=[-55.632, +48.792]
  ch7 (a_y   ): std=3.1640, range=[-50.835, +48.362]
  ch8 (a_z   ): std=2.2386, range=[-40.754, +42.999]

build_seq_t3, normalize_seq 정의 완료


## 8. GRU+F+W 모델 + 학습 함수

- 백본: 단방향 GRU (h=64, l=1)
- 메인 head: T+8 잔차 (회전 좌표계, tanh × 2cm)
- 보조 head F, W: λ=0.3 euclid loss


In [16]:
# ============================================================
# GRU 백본 (LSTM과 비슷한 구조, GRU cell 사용)
# 단일 fold 베스트: h=64, l=1 (0.6617 ± 0.0012)
# 학습 sweep 베스트: lr=5e-4, do=0.3 (트렌드 ↓)
# 5-fold OOF (sub_09): 0.6612 → LB 0.6778 (변환률 3.25 ★)
# ============================================================

class GRUModel(nn.Module):
    """GRU 시계열 인코더 + 메인 헤드."""
    def __init__(self, n_channels=9, scal_dim=40, hidden_size=64, num_layers=1,
                 bidirectional=False, fc_hidden=128, p=0.2, main_out_scale_cm=2.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_channels, hidden_size=hidden_size,
            num_layers=num_layers, bidirectional=bidirectional,
            batch_first=True, dropout=p if num_layers > 1 else 0,
        )
        gru_out = hidden_size * (2 if bidirectional else 1)
        self.fc1 = nn.Linear(gru_out + scal_dim, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden, fc_hidden // 2)
        self.head = nn.Linear(fc_hidden // 2, 3)
        self.act = nn.GELU()
        self.drop = nn.Dropout(p)
        self.main_out_scale_cm = main_out_scale_cm

    def forward(self, seq, scal):
        out, _ = self.gru(seq)
        x = out[:, -1, :]
        z = torch.cat([x, scal], dim=1)
        z = self.act(self.fc1(z)); z = self.drop(z)
        z = self.act(self.fc2(z))
        out = torch.tanh(self.head(z)) * (self.main_out_scale_cm / 100.0)
        return out

class GRUModelMultiAux(nn.Module):
    """GRU 시계열 인코더 + 메인 + 보조 헤드."""
    def __init__(self, n_channels=9, scal_dim=40, hidden_size=64, num_layers=1,
                 bidirectional=False, fc_hidden=128, p=0.2,
                 aux_dims=None, aux_clips=None, main_out_scale_cm=2.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_channels, hidden_size=hidden_size,
            num_layers=num_layers, bidirectional=bidirectional,
            batch_first=True, dropout=p if num_layers > 1 else 0,
        )
        gru_out = hidden_size * (2 if bidirectional else 1)
        self.fc1 = nn.Linear(gru_out + scal_dim, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden, fc_hidden // 2)
        self.act = nn.GELU()
        self.drop = nn.Dropout(p)
        self.head_main = nn.Linear(fc_hidden // 2, 3)
        self.aux_heads = nn.ModuleList([
            nn.Linear(fc_hidden // 2, d) for d in aux_dims
        ]) if aux_dims else nn.ModuleList([])
        self.aux_clips = aux_clips if aux_clips else [None] * len(aux_dims or [])
        self.main_out_scale_cm = main_out_scale_cm

    def forward(self, seq, scal):
        out, _ = self.gru(seq)
        x = out[:, -1, :]
        z = torch.cat([x, scal], dim=1)
        z = self.act(self.fc1(z)); z = self.drop(z)
        z = self.act(self.fc2(z))
        out_main = torch.tanh(self.head_main(z)) * (self.main_out_scale_cm / 100.0)
        outs_aux = []
        for h, clip in zip(self.aux_heads, self.aux_clips):
            o = h(z)
            if clip == 'tanh_2cm':
                o = torch.tanh(o) * 0.02
            outs_aux.append(o)
        return out_main, outs_aux

print('GRUModel + GRUModelMultiAux 정의 완료')


GRUModel + GRUModelMultiAux 정의 완료


In [17]:
# ============================================================
# 추가 학습 함수 (GRU / Transformer / Seq2Seq)
# BiLSTM은 train_lstm_main_only/train_lstm_multi_aux에 bidirectional=True 전달
# ============================================================

def train_gru_main_only(loss_fn, fold_data, *, epochs=200, batch=256, lr=1e-3,
                         weight_decay=1e-4, patience=30,
                         hidden_size=64, num_layers=1, bidirectional=False,
                         fc_hidden=128, dropout=0.2, seed=0):
    """GRU 메인 task 학습."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = GRUModel(
        n_channels=fold_data['seq_tr'].shape[2],
        hidden_size=hidden_size, num_layers=num_layers,
        bidirectional=bidirectional,
        scal_dim=fold_data['scal_tr'].shape[1],
        fc_hidden=fc_hidden, p=dropout,
    ).to(device)
    return _train_generic(model, loss_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=False)

def train_gru_multi_aux(loss_main_fn, aux_specs, fold_data,
                         *, epochs=200, batch=256, lr=1e-3,
                         weight_decay=1e-4, patience=30,
                         hidden_size=64, num_layers=1, bidirectional=False,
                         fc_hidden=128, dropout=0.2, seed=0):
    """GRU 멀티 헤드 학습."""
    torch.manual_seed(seed); np.random.seed(seed)
    aux_dims = [s['dim'] for s in aux_specs]
    aux_clips = [s['clip'] for s in aux_specs]
    model = GRUModelMultiAux(
        n_channels=fold_data['seq_tr'].shape[2],
        hidden_size=hidden_size, num_layers=num_layers,
        bidirectional=bidirectional,
        scal_dim=fold_data['scal_tr'].shape[1],
        fc_hidden=fc_hidden, p=dropout,
        aux_dims=aux_dims, aux_clips=aux_clips,
    ).to(device)
    return _train_generic(model, loss_main_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=True, aux_specs=aux_specs)

def train_transformer_main_only(loss_fn, fold_data, *, epochs=200, batch=256, lr=1e-3,
                                  weight_decay=1e-4, patience=30,
                                  d_model=64, nhead=4, num_layers=2, dim_feedforward=256,
                                  fc_hidden=128, dropout=0.2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CausalTransformerModel(
        n_channels=fold_data['seq_tr'].shape[2],
        scal_dim=fold_data['scal_tr'].shape[1],
        d_model=d_model, nhead=nhead, num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        fc_hidden=fc_hidden, p=dropout,
    ).to(device)
    return _train_generic(model, loss_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=False)

def train_transformer_multi_aux(loss_main_fn, aux_specs, fold_data,
                                  *, epochs=200, batch=256, lr=1e-3,
                                  weight_decay=1e-4, patience=30,
                                  d_model=64, nhead=4, num_layers=2, dim_feedforward=256,
                                  fc_hidden=128, dropout=0.2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    aux_dims = [s['dim'] for s in aux_specs]
    aux_clips = [s['clip'] for s in aux_specs]
    model = CausalTransformerModelMultiAux(
        n_channels=fold_data['seq_tr'].shape[2],
        scal_dim=fold_data['scal_tr'].shape[1],
        d_model=d_model, nhead=nhead, num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        fc_hidden=fc_hidden, p=dropout,
        aux_dims=aux_dims, aux_clips=aux_clips,
    ).to(device)
    return _train_generic(model, loss_main_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=True, aux_specs=aux_specs)

def train_seq2seq_main_only(loss_fn, fold_data, *, epochs=200, batch=256, lr=1e-3,
                              weight_decay=1e-4, patience=30,
                              hidden_size=64, num_layers=1,
                              fc_hidden=128, dropout=0.2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LSTMSeq2Seq(
        n_channels=fold_data['seq_tr'].shape[2],
        hidden_size=hidden_size, num_layers=num_layers,
        scal_dim=fold_data['scal_tr'].shape[1],
        fc_hidden=fc_hidden, p=dropout,
    ).to(device)
    return _train_generic(model, loss_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=False)

def train_seq2seq_multi_aux(loss_main_fn, aux_specs, fold_data,
                              *, epochs=200, batch=256, lr=1e-3,
                              weight_decay=1e-4, patience=30,
                              hidden_size=64, num_layers=1,
                              fc_hidden=128, dropout=0.2, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    aux_dims = [s['dim'] for s in aux_specs]
    aux_clips = [s['clip'] for s in aux_specs]
    model = LSTMSeq2SeqMultiAux(
        n_channels=fold_data['seq_tr'].shape[2],
        hidden_size=hidden_size, num_layers=num_layers,
        scal_dim=fold_data['scal_tr'].shape[1],
        fc_hidden=fc_hidden, p=dropout,
        aux_dims=aux_dims, aux_clips=aux_clips,
    ).to(device)
    return _train_generic(model, loss_main_fn, fold_data, epochs, batch, lr,
                          weight_decay, patience, multi_aux=True, aux_specs=aux_specs)

def _train_generic(model, loss_fn, fold_data, epochs, batch, lr,
                    weight_decay, patience, multi_aux=False, aux_specs=None):
    """공통 학습 루프."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    def T(a): return torch.from_numpy(a).to(device)
    seq_tr = T(fold_data['seq_tr']); scal_tr = T(fold_data['scal_tr'])
    tgt_tr = T(fold_data['tgt_tr_rot'])
    seq_va = T(fold_data['seq_va']); scal_va = T(fold_data['scal_va'])
    if multi_aux:
        aux_targets = [T(s['target_tr'].astype(np.float32)) for s in aux_specs]
    best_rh, best_state, no_improve = -1, None, 0
    n_tr = seq_tr.shape[0]
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_tr, device=device)
        for i in range(0, n_tr, batch):
            idx = perm[i:i+batch]
            opt.zero_grad()
            if multi_aux:
                out_main, outs_aux = model(seq_tr[idx], scal_tr[idx])
                loss = loss_fn(out_main, tgt_tr[idx])
                for k, sp in enumerate(aux_specs):
                    loss = loss + sp['lambda'] * sp['loss_fn'](outs_aux[k], aux_targets[k][idx])
            else:
                out = model(seq_tr[idx], scal_tr[idx])
                loss = loss_fn(out, tgt_tr[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            if multi_aux:
                out_va_rot, _ = model(seq_va, scal_va)
                out_va_rot = out_va_rot.cpu().numpy()
            else:
                out_va_rot = model(seq_va, scal_va).cpu().numpy()
        out_va = inverse_rotate_xy(out_va_rot, fold_data['theta_va'])
        pred = fold_data['kal_va'] + out_va
        rh = float((np.linalg.norm(pred - fold_data['y_va'], axis=-1) <= 0.01).mean())
        if rh > best_rh:
            best_rh = rh
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    return best_rh

print('GRU/Trans/Seq2Seq 학습 함수 정의 완료')
print('  BiLSTM은 train_lstm_main_only/train_lstm_multi_aux에 bidirectional=True 전달')


GRU/Trans/Seq2Seq 학습 함수 정의 완료
  BiLSTM은 train_lstm_main_only/train_lstm_multi_aux에 bidirectional=True 전달


## 9. 5-fold OOF + test 예측 runner (`run_gru_fw_5fold`)


In [18]:
# ============================================================
# 5-fold OOF runner 함수 (백본별 — generic 함수로 추상화도 가능)
# run_gru_fw_5fold / run_transformer_fw_5fold / run_seq2seq_main_5fold / run_seq2seq_fw_5fold
# run_bilstm_fw_5fold는 run_lstm_fw_5fold에 bidirectional=True 전달
# ============================================================

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

def _run_backbone_5fold(model_class, model_class_aux,
                          target_main, target_F, target_W,
                          seq_arr, scal_arr, seq_te, scal_te,
                          config, multi_aux=True,
                          n_seeds=3, n_folds=5, verbose=True,
                          lambda_F=0.3, lambda_W=0.3):
    """Generic 5-fold OOF + test 예측 runner."""
    oof_rot = np.zeros((len(X_train), 3))
    test_per_fold = []
    fold_rh = []

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    t_start = time.time()

    for fold_i, (tr_idx, va_idx) in enumerate(kf.split(np.arange(len(X_train)))):
        sc_seq_fold = StandardScaler().fit(seq_arr[tr_idx].reshape(-1, seq_arr.shape[2]))
        sc_scal_fold = StandardScaler().fit(scal_arr[tr_idx])

        seq_tr_n = normalize_seq(seq_arr[tr_idx], sc_seq_fold)
        seq_va_n = normalize_seq(seq_arr[va_idx], sc_seq_fold)
        seq_te_n = normalize_seq(seq_te, sc_seq_fold)

        scal_te_t = torch.from_numpy(sc_scal_fold.transform(scal_te).astype(np.float32)).to(device)
        seq_te_t = torch.from_numpy(seq_te_n).to(device)

        fd = {
            'seq_tr': seq_tr_n,
            'scal_tr': sc_scal_fold.transform(scal_arr[tr_idx]).astype(np.float32),
            'tgt_tr_rot': target_main[tr_idx].astype(np.float32),
            'seq_va': seq_va_n,
            'scal_va': sc_scal_fold.transform(scal_arr[va_idx]).astype(np.float32),
            'kal_va': kalman_train[va_idx],
            'theta_va': theta_train[va_idx],
            'y_va': y_train[va_idx],
        }

        if multi_aux:
            aux_specs = [
                {'name': 'F', 'target_tr': target_F[tr_idx], 'loss_fn': loss_aux_euclid,
                 'lambda': lambda_F, 'dim': 3, 'clip': None},
                {'name': 'W', 'target_tr': target_W[tr_idx], 'loss_fn': loss_aux_euclid,
                 'lambda': lambda_W, 'dim': 3, 'clip': None},
            ]

        seed_val_rot, seed_test_rot = [], []
        for s in range(n_seeds):
            torch.manual_seed(s); np.random.seed(s)

            if multi_aux:
                aux_dims = [sp['dim'] for sp in aux_specs]
                aux_clips = [sp['clip'] for sp in aux_specs]
                model = model_class_aux(
                    n_channels=fd['seq_tr'].shape[2],
                    scal_dim=fd['scal_tr'].shape[1],
                    **{k: v for k, v in config.items() if k not in ['lr', 'wd']},
                    aux_dims=aux_dims, aux_clips=aux_clips,
                ).to(device)
            else:
                model = model_class(
                    n_channels=fd['seq_tr'].shape[2],
                    scal_dim=fd['scal_tr'].shape[1],
                    **{k: v for k, v in config.items() if k not in ['lr', 'wd']},
                ).to(device)

            opt = torch.optim.AdamW(model.parameters(),
                                    lr=config['lr'], weight_decay=config['wd'])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=200)

            def T(a): return torch.from_numpy(a).to(device)
            seq_tr = T(fd['seq_tr']); scal_tr = T(fd['scal_tr'])
            tgt_tr = T(fd['tgt_tr_rot'])
            seq_va = T(fd['seq_va']); scal_va = T(fd['scal_va'])
            if multi_aux:
                aux_targets = [T(sp['target_tr'].astype(np.float32)) for sp in aux_specs]

            best_rh, best_state, no_improve = -1, None, 0
            n_tr = seq_tr.shape[0]
            for ep in range(200):
                model.train()
                perm = torch.randperm(n_tr, device=device)
                for j in range(0, n_tr, 256):
                    idx = perm[j:j+256]
                    opt.zero_grad()
                    if multi_aux:
                        out_main, outs_aux = model(seq_tr[idx], scal_tr[idx])
                        loss = LOSSES_V2['combo'](out_main, tgt_tr[idx])
                        for k, sp in enumerate(aux_specs):
                            loss = loss + sp['lambda'] * sp['loss_fn'](outs_aux[k], aux_targets[k][idx])
                    else:
                        out = model(seq_tr[idx], scal_tr[idx])
                        loss = LOSSES_V2['combo'](out, tgt_tr[idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sched.step()
                model.eval()
                with torch.no_grad():
                    if multi_aux:
                        out_va_rot, _ = model(seq_va, scal_va)
                        out_va_rot = out_va_rot.cpu().numpy()
                    else:
                        out_va_rot = model(seq_va, scal_va).cpu().numpy()
                out_va = inverse_rotate_xy(out_va_rot, fd['theta_va'])
                pred = fd['kal_va'] + out_va
                rh = float((np.linalg.norm(pred - fd['y_va'], axis=-1) <= 0.01).mean())
                if rh > best_rh:
                    best_rh = rh
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= 30:
                    break

            model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                if multi_aux:
                    pred_va_rot, _ = model(seq_va, scal_va)
                    pred_te_rot, _ = model(seq_te_t, scal_te_t)
                    pred_va_rot = pred_va_rot.cpu().numpy()
                    pred_te_rot = pred_te_rot.cpu().numpy()
                else:
                    pred_va_rot = model(seq_va, scal_va).cpu().numpy()
                    pred_te_rot = model(seq_te_t, scal_te_t).cpu().numpy()
            seed_val_rot.append(pred_va_rot)
            seed_test_rot.append(pred_te_rot)
            torch.cuda.empty_cache(); gc.collect()

        val_mean_rot = np.mean(seed_val_rot, axis=0)
        test_mean_rot = np.mean(seed_test_rot, axis=0)
        oof_rot[va_idx] = val_mean_rot
        test_per_fold.append(test_mean_rot)

        val_unrot = inverse_rotate_xy(val_mean_rot, theta_train[va_idx])
        pred_pos = kalman_train[va_idx] + val_unrot
        rh_fold = float((np.linalg.norm(pred_pos - y_train[va_idx], axis=-1) <= 0.01).mean())
        fold_rh.append(rh_fold)
        if verbose:
            print(f'  fold {fold_i+1}/{n_folds}: R-Hit={rh_fold:.4f}')

    oof_unrot = inverse_rotate_xy(oof_rot, theta_train)
    pred_oof = kalman_train + oof_unrot
    oof_rhit = float((np.linalg.norm(pred_oof - y_train, axis=-1) <= 0.01).mean())
    test_unrot_folds = [inverse_rotate_xy(rot, theta_test) for rot in test_per_fold]
    test_unrot = np.mean(test_unrot_folds, axis=0)

    if verbose:
        print(f'  OOF R-Hit: {oof_rhit:.4f}  '
              f'(fold mean ± std: {np.mean(fold_rh):.4f} ± {np.std(fold_rh):.4f})')
        print(f'  소요: {(time.time()-t_start)/60:.1f}분')

    return oof_unrot, test_unrot, fold_rh, oof_rhit

# 각 백본별 wrapper
def run_gru_fw_5fold(target_main, target_F, target_W,
                      seq_arr, scal_arr, seq_te, scal_te, config, **kwargs):
    return _run_backbone_5fold(GRUModel, GRUModelMultiAux,
                                target_main, target_F, target_W,
                                seq_arr, scal_arr, seq_te, scal_te,
                                config, multi_aux=True, **kwargs)

def run_transformer_fw_5fold(target_main, target_F, target_W,
                               seq_arr, scal_arr, seq_te, scal_te, config, **kwargs):
    return _run_backbone_5fold(CausalTransformerModel, CausalTransformerModelMultiAux,
                                target_main, target_F, target_W,
                                seq_arr, scal_arr, seq_te, scal_te,
                                config, multi_aux=True, **kwargs)

def run_bilstm_fw_5fold(target_main, target_F, target_W,
                         seq_arr, scal_arr, seq_te, scal_te, config, **kwargs):
    # LSTMModel/MultiAux의 bidirectional=True 활용
    cfg = {**config, 'bidirectional': True}
    return _run_backbone_5fold(LSTMModel, LSTMModelMultiAux,
                                target_main, target_F, target_W,
                                seq_arr, scal_arr, seq_te, scal_te,
                                cfg, multi_aux=True, **kwargs)

def run_seq2seq_main_5fold(target_main, target_F, target_W,
                            seq_arr, scal_arr, seq_te, scal_te, config, **kwargs):
    return _run_backbone_5fold(LSTMSeq2Seq, None,
                                target_main, target_F, target_W,
                                seq_arr, scal_arr, seq_te, scal_te,
                                config, multi_aux=False, **kwargs)

def run_seq2seq_fw_5fold(target_main, target_F, target_W,
                          seq_arr, scal_arr, seq_te, scal_te, config, **kwargs):
    return _run_backbone_5fold(None, LSTMSeq2SeqMultiAux,
                                target_main, target_F, target_W,
                                seq_arr, scal_arr, seq_te, scal_te,
                                config, multi_aux=True, **kwargs)

print('5-fold OOF runner 정의 완료:')
print('  run_gru_fw_5fold, run_transformer_fw_5fold, run_bilstm_fw_5fold')
print('  run_seq2seq_main_5fold, run_seq2seq_fw_5fold')


5-fold OOF runner 정의 완료:
  run_gru_fw_5fold, run_transformer_fw_5fold, run_bilstm_fw_5fold
  run_seq2seq_main_5fold, run_seq2seq_fw_5fold


## 10. sub_09 두 셋업 5-fold OOF + 평균

sub_09 = 두 셋업 (lr=5e-4 do=0.3 + lr=1e-3 do=0.1)의 OOF/test 평균.

- 셋업 A: 안정적 (학습 sweep best)
- 셋업 B: 단일 fold best (공격적, 더 큰 LR + 작은 dropout)
- 두 셋업 평균이 LB 0.6778 결정.


In [22]:
# ============================================================
# sub_09 두 셋업 5-fold OOF + test 예측
# ============================================================

import time
import gc

GRU_CONFIG_A = {  # 안정적 (학습 sweep best)
    'hidden_size': 64, 'num_layers': 1, 'fc_hidden': 128,
    'lr': 5e-4, 'p': 0.3, 'wd': 1e-4,
}

GRU_CONFIG_B = {  # 공격적 (단일 fold best)
    'hidden_size': 64, 'num_layers': 1, 'fc_hidden': 128,
    'lr': 1e-3, 'p': 0.1, 'wd': 1e-4,
}

print("=" * 60)
print("sub_09 셋업 A 학습 (lr=5e-4, do=0.3)")
print("=" * 60)
oof_A, test_A, fold_rh_A, oof_rhit_A = run_gru_fw_5fold(
    target_T8, aux_target_F, aux_target_W,
    seq_train, X_scal_train_t3, seq_test, X_scal_test_t3,
    config=GRU_CONFIG_A,
)
print(f"\n셋업 A OOF: {oof_rhit_A:.4f}")

print("\n" + "=" * 60)
print("sub_09 셋업 B 학습 (lr=1e-3, do=0.1)")
print("=" * 60)
oof_B, test_B, fold_rh_B, oof_rhit_B = run_gru_fw_5fold(
    target_T8, aux_target_F, aux_target_W,
    seq_train, X_scal_train_t3, seq_test, X_scal_test_t3,
    config=GRU_CONFIG_B,
)
print(f"\n셋업 B OOF: {oof_rhit_B:.4f}")

# sub_09 = 두 셋업 평균
oof_sub09  = (oof_A + oof_B) / 2
test_sub09 = (test_A + test_B) / 2

# OOF 검증
pred_oof_sub09 = kalman_train + oof_sub09
rh_sub09 = float((np.linalg.norm(pred_oof_sub09 - y_train, axis=-1) <= 0.01).mean())
print(f"\n=== sub_09 (두 셋업 평균) ===")
print(f"  OOF R-Hit: {rh_sub09:.4f}  (기대: 0.6612)")
print(f"  LB (기록): 0.6778")


print(f"\n★ 원본 sub_09 OOF 0.6612 — 현재 {rh_sub09:.4f}, 차이 {abs(rh_sub09 - 0.6612):.4f}")
print(f"★ 차이가 0.002 이상이면 결정성 모드 fluctuation. 셀 12의 USE_DRIVE_BACKUP=True로 안전 재현")


sub_09 셋업 A 학습 (lr=5e-4, do=0.3)
  fold 1/5: R-Hit=0.6635
  fold 2/5: R-Hit=0.6460
  fold 3/5: R-Hit=0.6555
  fold 4/5: R-Hit=0.6715
  fold 5/5: R-Hit=0.6590
  OOF R-Hit: 0.6591  (fold mean ± std: 0.6591 ± 0.0085)
  소요: 5.6분

셋업 A OOF: 0.6591

sub_09 셋업 B 학습 (lr=1e-3, do=0.1)
  fold 1/5: R-Hit=0.6640
  fold 2/5: R-Hit=0.6500
  fold 3/5: R-Hit=0.6590
  fold 4/5: R-Hit=0.6745
  fold 5/5: R-Hit=0.6585
  OOF R-Hit: 0.6612  (fold mean ± std: 0.6612 ± 0.0080)
  소요: 3.8분

셋업 B OOF: 0.6612

=== sub_09 (두 셋업 평균) ===
  OOF R-Hit: 0.6598  (기대: 0.6612)
  LB (기록): 0.6778

★ 원본 sub_09 OOF 0.6612 — 현재 0.6598, 차이 0.0014
★ 차이가 0.002 이상이면 결정성 모드 fluctuation. 셀 12의 USE_DRIVE_BACKUP=True로 안전 재현


## 11. Calibration — Per-axis α 하드코드 (v2)

★ **v2 핵심 변경**: grid search 제거, 원본 best α 직접 적용.

### v1 문제
- Grid search가 학습 fluctuation에 sensitive
- OOF 다르면 다른 α 발견 → test에 적용 시 회귀
- 5-fold CV 음수 (-0.0008), overfit gap +0.0021
- v1 재현: α (0.9, 0.825, 1) 발견 → LB 0.6756

### v2 솔루션
- 원본 sub_24 best α **(1.000, 0.950, 1.000)** 하드코드
- y axis 5% 축소 (systematic bias 가설)
- Grid search overfit 완전 회피


In [23]:
# ============================================================
# Calibration — Per-axis α 하드코드
# ============================================================

# ★ 원본 sub_24 best α
BEST_ALPHAS = [1.000, 0.950, 1.000]

def rhit_from_residual(residual, kalman, y):
    pred = kalman + residual
    return float((np.linalg.norm(pred - y, axis=-1) <= 0.01).mean())

oof_base = oof_sub09
test_base = test_sub09

base_rh = rhit_from_residual(oof_base, kalman_train, y_train)
print(f"Base OOF R-Hit (sub_09): {base_rh:.4f}")

alpha_arr = np.array(BEST_ALPHAS)[None, :]
oof_calibrated = oof_base * alpha_arr
test_calibrated = test_base * alpha_arr

cal_rh = rhit_from_residual(oof_calibrated, kalman_train, y_train)
print(f"\n=== Calibration 적용 ===")
print(f"  α: ({BEST_ALPHAS[0]:.3f}, {BEST_ALPHAS[1]:.3f}, {BEST_ALPHAS[2]:.3f})")
print(f"  OOF R-Hit: {cal_rh:.4f}  (Δ {cal_rh - base_rh:+.4f})")
print(f"  Test residual mean norm: {np.linalg.norm(test_calibrated, axis=-1).mean()*100:.3f}cm")


Base OOF R-Hit (sub_09): 0.6598

=== Calibration 적용 ===
  α: (1.000, 0.950, 1.000)
  OOF R-Hit: 0.6600  (Δ +0.0002)
  Test residual mean norm: 0.361cm


## 12. Drive 백업 cross-check + sub_24 제출

★ **v2 핵심 안전망**: 학습 결과 vs Drive 백업 비교 → 100% 재현 옵션.

`USE_DRIVE_BACKUP` 플래그:
- **True (권고)**: Drive 백업의 `calibrated_sub09` 사용 → **LB 0.6780 100% 재현**
- **False**: 학습 결과 + 하드코드 α 사용 → LB 0.6776~0.6780 (학습 fluctuation 따라)


In [24]:
# ============================================================
# sub_24 제출 — Drive cross-check + fallback
# ============================================================

# ★ 모드 선택
USE_DRIVE_BACKUP = True   # True: 100% 재현 (LB 0.6780), False: 학습 결과 사용

DOCS_DIR = "/content/drive/MyDrive/competition/mosquito/docs"

# Drive 백업 로드
try:
    backup_test = dict(np.load(f"{DOCS_DIR}/test_residuals_v3.npz"))
    has_backup = "calibrated_sub09" in backup_test
    print(f"Drive 백업 로드 완료. calibrated_sub09 존재: {has_backup}")
except Exception as e:
    has_backup = False
    print(f"Drive 백업 로드 실패: {e}")

# Cross-check
if has_backup:
    backup_cal = backup_test["calibrated_sub09"]
    diff = np.linalg.norm(test_calibrated - backup_cal, axis=-1)
    print(f"\n=== Cross-check (학습 vs Drive 백업) ===")
    print(f"  평균 거리:  {diff.mean()*1000:.3f} mm")
    print(f"  최대 거리:  {diff.max()*1000:.3f} mm")
    print(f"  p95 거리:   {np.percentile(diff, 95)*1000:.3f} mm")
    if diff.mean() < 1e-9:
        print(f"  ★ 완전 일치! (결정성 모드 성공)")
    elif diff.mean() < 1e-4:
        print(f"  거의 일치 (작은 fluctuation, LB ≈ 동일)")
    else:
        print(f"  학습이 백업과 다름 (USE_DRIVE_BACKUP=True 권고)")

# 최종 prediction 결정
if USE_DRIVE_BACKUP and has_backup:
    test_final = backup_cal
    print(f"\n★ Drive 백업 사용 — LB 0.6780 100% 재현")
else:
    test_final = test_calibrated
    print(f"\n★ 학습 결과 사용 — LB 추정 0.6776~0.6780")

# 제출 파일
test_pred_final = kalman_test + test_final

sub_final = pd.DataFrame({
    "id": sub["id"].tolist(),
    "x":  test_pred_final[:, 0],
    "y":  test_pred_final[:, 1],
    "z":  test_pred_final[:, 2],
})

DRIVE_DIR = "/content/drive/MyDrive/competition/mosquito/submissions"
os.makedirs(DRIVE_DIR, exist_ok=True)
path_local = "/content/sub_24_final_v2.csv"
path_drive = f"{DRIVE_DIR}/sub_24_final_v2.csv"
sub_final.to_csv(path_local, index=False)
sub_final.to_csv(path_drive, index=False)

mode = "Drive 백업 (100% 재현)" if (USE_DRIVE_BACKUP and has_backup) else "학습 + α 하드코드"
print(f"\n" + "=" * 60)
print(f"sub_24 v2 제출 완료")
print(f"=" * 60)
print(f"  모드:       {mode}")
print(f"  Local:      {path_local}")
print(f"  Drive:      {path_drive}")
print(f"  α:          (1.000, 0.950, 1.000) — 하드코드")
print(f"  기대 LB:    0.6780 ★")


Drive 백업 로드 완료. calibrated_sub09 존재: True

=== Cross-check (학습 vs Drive 백업) ===
  평균 거리:  0.267 mm
  최대 거리:  2.428 mm
  p95 거리:   0.658 mm
  학습이 백업과 다름 (USE_DRIVE_BACKUP=True 권고)

★ Drive 백업 사용 — LB 0.6780 100% 재현

sub_24 v2 제출 완료
  모드:       Drive 백업 (100% 재현)
  Local:      /content/sub_24_final_v2.csv
  Drive:      /content/drive/MyDrive/competition/mosquito/submissions/sub_24_final_v2.csv
  α:          (1.000, 0.950, 1.000) — 하드코드
  기대 LB:    0.6780 ★


## 13. 최종 결과 요약 (v2)

### v2 변경 정리
1. 결정성 모드 — torch/cudnn/numpy/random seed 고정 (v1 미적용)
2. Calibration α 하드코드 (1, 0.95, 1) — grid search 완전 제거
3. Drive 백업 cross-check + fallback — 100% 재현 안전망

### 솔직한 한계
- 결정성 모드 적용해도 **원본 학습 시 비결정적이었으면 ±0.001~0.003 fluctuation 가능**
- v6 노트북에 결정성 설정 없음 → 원본 학습이 비결정적이었을 가능성 큼
- 따라서 진정 "완전 동일" 재현 = `USE_DRIVE_BACKUP = True`

### 점수 비교
| 시도 | OOF | LB | 비고 |
|---|---|---|---|
| Kalman baseline | - | 0.6452 | CV |
| sub_09 (GRU+F+W 두 셋업) | 0.6612 | 0.6778 | RNN 단방향 |
| **sub_24 (sub_09 + α=0.95)** ★ | 0.6625 | **0.6780** | 19 LB 중 최고 |
| v1 재현 시도 | 0.6598 | 0.6756 | grid search overfit |
| **v2 (이 노트북, USE_DRIVE_BACKUP=True)** | - | **0.6780** ★ | 백업 사용, 100% 재현 |
